## Q1. Embedding a query

Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [14]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"

q1 = embed.encode(query)

q1[0]

np.float64(-0.02058203437252893)

## Q2. Cosine similarity

In [15]:
from ingest import load_lessons_data

documents = load_lessons_data()

In [16]:
item = next((doc["content"] for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"), None)        

In [17]:
dv = embed.encode(item)
q1.dot(dv)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand

In [7]:
len(documents)

72

In [42]:
from gitsource import chunk_documents
import numpy as np


chunks = chunk_documents(documents, size=2000, step=1000)
texts = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(texts)
scores = X.dot(q1)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489017718578813))

In [41]:
chunks[94]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

In [51]:
from minsearch import VectorSearch

ques = "What metric do we use to evaluate a search engine?"
qv = embed.encode(ques)

v_index = VectorSearch(
    keyword_fields=["filename"]
)
v_index.fit(X, chunks)
results = v_index.search(qv, num_results=5)

In [52]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [54]:
from minsearch import Index

q5 = "How do I store vectors in PostgreSQL?"
q5v = embed.encode(q5)

k_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
k_index.fit(chunks)

In [57]:
k_search_results = k_index.search(q5, num_results=5, boost_dict={"content":1})
k_search_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [58]:
v_search_results = v_index.search(q5v, num_results=5)
v_search_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

Ans:  02-vector-search/lessons/08-pgvector.md

## 6. Hybrid search

In [62]:
question = "How do I give the model access to tools?"
qv = embed.encode(question)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

vector_results = v_index.search(qv, num_results=5)

text_results = k_index.search(question, num_results=5, boost_dict={"content":1})

results = rrf([vector_results, text_results])

In [64]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'